# Demo 05b — Neural Networks with Keras

**Class 05 · Blocks 5-8** (slides: `slides/05_svm_neural_networks.md`)

Six live demos. The networks are built with **Keras 3** (the JAX backend); one demo shows the same network in plain JAX
(the practice exercise of the lab).

* **H1 — Neurons, layers and activations:** XOR, the number of hidden units, the universal-approximation picture.
* **H2 — Backpropagation:** a 2-2-1 network by hand, checked against `jax.grad`; vanishing gradients; counting parameters.
* **H3 — Training:** learning rate, batch size, optimizers, initialization, seeds; the same network in plain JAX.
* **H4 — Capacity and regularization:** overfitting, weight decay, dropout, early stopping.
* **H5 — Regression with a network** (concrete strength, California housing).
* **H6 — Which learner?** Logistic regression, SVM and MLP on five datasets, with a paired test.

Run time: about 4 minutes in total on a laptop CPU, no cell above one minute (the slowest are the multi-seed tables).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import time
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

jax.config.update("jax_enable_x64", False)       # Keras works in float32
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")
print("Keras", keras.__version__, "backend:", keras.backend.backend())


pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")


def table(rows, cols):
    """Print a list of rows as a polars table (hyperparameters such as C and gamma are written with `:g`)."""
    keep = {"C", "gamma", "epsilon", "learning rate"}
    rows = [tuple(f"{v:g}" if c in keep else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: nulls -> median (numbers) or "missing" (text), text -> one-hot."""
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy()


load_xy = load_table


def mlp(sizes, n_in, n_out, activation="relu", out="softmax", l2=0.0, dropout=0.0, init="glorot_uniform", seed=0):
    """A Keras multilayer perceptron: hidden layers `sizes`, an output layer of `n_out` units."""
    keras.utils.set_random_seed(seed)
    reg = keras.regularizers.L2(l2) if l2 else None
    layers = [keras.Input((n_in,))]
    for s in sizes:
        layers.append(keras.layers.Dense(s, activation=activation, kernel_regularizer=reg, kernel_initializer=init))
        if dropout:
            layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(n_out, activation=out, kernel_initializer=init))
    return keras.Sequential(layers)


def fit_classifier(model, X, y, Xv=None, yv=None, epochs=60, lr=1e-3, batch=32, opt="adam", callbacks=None):
    loss = "binary_crossentropy" if model.output_shape[-1] == 1 else "sparse_categorical_crossentropy"
    optimizer = {"adam": keras.optimizers.Adam, "sgd": keras.optimizers.SGD, "rmsprop": keras.optimizers.RMSprop}[opt](lr)
    model.compile(optimizer, loss, metrics=["accuracy"])
    val = (Xv, yv) if Xv is not None else None
    return model.fit(X, y, epochs=epochs, batch_size=batch, validation_data=val, verbose=0, callbacks=callbacks)


def accuracy(model, X, y):
    p = model.predict(X, verbose=0)
    pred = (p[:, 0] > 0.5).astype(int) if p.shape[1] == 1 else p.argmax(1)
    return float(np.mean(pred == y))


def boundary(ax, predict, X, y, title=""):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200), np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
    z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).reshape(gx.shape)
    ax.contourf(gx, gy, z, levels=[0, 0.5, 1], alpha=0.15, cmap="coolwarm")
    ax.contour(gx, gy, z, levels=[0.5], colors="k", linewidths=1)
    ax.scatter(*X.T, c=y, cmap="coolwarm", s=14, edgecolor="k", linewidth=0.3)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

## H1 — Neurons, layers and activations

An **artificial neuron** computes a weighted sum of its inputs and passes it through an **activation function**:
$a = \phi(w^\top x + b)$. With $\phi$ = identity it is linear regression; with the sigmoid it is **logistic regression**; with a
step it is the **perceptron** of Class 03. A **layer** is many neurons reading the same inputs, written as a matrix product:
$a = \phi(W x + b)$. A **multilayer perceptron (MLP)** stacks layers: the output of one layer is the input of the next.

The activation must be **non-linear**: two linear layers compose into one linear layer, so depth would add nothing.

In [ ]:
z = np.linspace(-4, 4, 400)
acts = {
    "sigmoid": (1 / (1 + np.exp(-z)), None),
    "tanh": (np.tanh(z), None),
    "ReLU": (np.maximum(0, z), None),
    "leaky ReLU (0.1)": (np.where(z > 0, z, 0.1 * z), None),
    "softplus": (np.log1p(np.exp(z)), None),
    "GELU": (0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z**3))), None),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for name, (a, _) in acts.items():
    axes[0].plot(z, a, label=name)
    axes[1].plot(z[1:], np.diff(a) / np.diff(z), label=name)
axes[0].set(title="activation $\\phi(z)$", ylim=(-1.5, 4))
axes[1].set(title="derivative $\\phi'(z)$", ylim=(-0.3, 1.3))
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("largest derivative:  sigmoid", 0.25, "  tanh", 1.0, "  ReLU", 1.0, "(sigmoid derivative <= 1/4: it shrinks every gradient it touches)")
A_, B_, v_ = np.random.default_rng(0).normal(size=(3, 4)), np.random.default_rng(1).normal(size=(4, 2)), np.array([1.0, -2.0])
print("two linear layers are one linear layer: A (B v) == (A B) v ->", np.allclose(A_ @ (B_ @ v_), (A_ @ B_) @ v_))

### XOR: what a single neuron cannot do

XOR is 1 when exactly one input is 1. No line separates $\{(0,1),(1,0)\}$ from $\{(0,0),(1,1)\}$ (Class 03). One hidden layer
of two neurons solves it, with weights we can write by hand:
$h_1 = \text{ReLU}(x_1 + x_2)$, $h_2 = \text{ReLU}(x_1 + x_2 - 1)$, output $= h_1 - 2 h_2$.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_xor = np.array([0, 1, 1, 0])
def relu(a):
    return np.maximum(0, a)


W1, b1 = np.array([[1.0, 1.0], [1.0, 1.0]]), np.array([0.0, -1.0])
W2, b2 = np.array([1.0, -2.0]), 0.0
h = relu(X_xor @ W1 + b1)
out = h @ W2 + b2
print(f"{'x1':>3} {'x2':>3} | {'h1':>4} {'h2':>4} | {'output':>6} {'target':>6}")
for xi, hi, oi, ti in zip(X_xor, h, out, y_xor):
    print(f"{xi[0]:>3.0f} {xi[1]:>3.0f} | {hi[0]:>4.0f} {hi[1]:>4.0f} | {oi:>6.0f} {ti:>6d}")

# a single neuron on the XOR cloud of Class 03 versus one hidden layer, trained with Keras
rng = np.random.default_rng(0)
Xx = rng.uniform(0.15, 1.0, (400, 2)) * rng.choice([-1.0, 1.0], (400, 2))
yx = (Xx[:, 0] * Xx[:, 1] < 0).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (sizes, label) in zip(axes, (([], "single neuron (logistic)"), ([2], "2 hidden units"), ([8], "8 hidden units"))):
    net = mlp(sizes, 2, 1, activation="relu", out="sigmoid", seed=1)
    fit_classifier(net, Xx, yx, epochs=150, lr=0.05, batch=40)
    boundary(ax, lambda P, net=net: net.predict(P, verbose=0)[:, 0], Xx, yx, f"{label}: accuracy {accuracy(net, Xx, yx):.2f}")
plt.tight_layout()
plt.show()

**How many hidden units?** Moons (200 points, noise 0.25), a network with one hidden layer of $H$ ReLU units. Median over 3
seeds, test accuracy on 2000 fresh points. The boundary is a **piecewise-linear** curve with about $H$ pieces.

In [ ]:
Xm, ym = make_moons(200, noise=0.25, random_state=0)
Xm_te, ym_te = make_moons(2000, noise=0.25, random_state=1)
units = [1, 2, 3, 5, 10, 50]
fig, axes = plt.subplots(1, len(units), figsize=(16, 2.9))
rows = []
for ax, H in zip(axes, units):
    accs, last = [], None
    for seed in range(3):
        net = mlp([H], 2, 1, out="sigmoid", seed=seed)
        fit_classifier(net, Xm, ym, epochs=120, lr=0.02, batch=32)
        accs.append((accuracy(net, Xm, ym), accuracy(net, Xm_te, ym_te)))
        last = net
    boundary(ax, lambda P, net=last: net.predict(P, verbose=0)[:, 0], Xm, ym, f"H = {H}")
    a = np.array(accs)
    rows.append((H, last.count_params(), float(np.median(a[:, 0])), float(np.median(a[:, 1]))))
plt.tight_layout()
plt.show()
table(rows, ["hidden units", "parameters", "train acc", "test acc (median of 3 seeds)"])

**The activation matters.** The same network (two hidden layers of 16 units) with four activations on the moons data and on
the digits. A *linear* activation collapses to logistic regression.

In [ ]:
Xd, yd = load_xy("digits", "target")
Xd = Xd / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, stratify=yd, random_state=0)
rows = []
for act in ("linear", "sigmoid", "tanh", "relu"):
    am, ad = [], []
    for seed in range(2):
        net = mlp([16, 16], 2, 1, activation=act, out="sigmoid", seed=seed)
        fit_classifier(net, Xm, ym, epochs=100, lr=0.01, batch=32)
        am.append(accuracy(net, Xm_te, ym_te))
        net = mlp([16, 16], 64, 10, activation=act, seed=seed)
        fit_classifier(net, Xd_tr, yd_tr, epochs=20, lr=0.005, batch=64)
        ad.append(accuracy(net, Xd_te, yd_te))
    rows.append((act, float(np.median(am)), float(np.median(ad))))
table(rows, ["activation", "moons test acc", "digits test acc"])
lr_m = LogisticRegression().fit(Xm, ym)
print(f"logistic regression: moons {lr_m.score(Xm_te, ym_te):.3f}")

**Universal approximation, seen.** One hidden layer with enough ReLU units can follow *any* continuous curve on a bounded
interval. Fit $\sin(2x)$ with 2, 5, 20 and 100 units: each unit adds a "kink"; the fit improves with $H$.

In [ ]:
xs = np.linspace(-3, 3, 300)[:, None].astype("float32")
ys = np.sin(2 * xs[:, 0]).astype("float32")
fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
for ax, H in zip(axes, (2, 5, 20, 100)):
    net = mlp([H], 1, 1, out=None, seed=0)
    net.compile(keras.optimizers.Adam(0.01), "mse")
    net.fit(xs, ys, epochs=400, batch_size=64, verbose=0)
    f = net.predict(xs, verbose=0)[:, 0]
    ax.plot(xs[:, 0], ys, "k:")
    ax.plot(xs[:, 0], f, color="tab:red")
    ax.set_title(f"H = {H}: RMSE {np.sqrt(np.mean((f - ys) ** 2)):.3f}", fontsize=9)
plt.tight_layout()
plt.show()

## H2 — Backpropagation

Training needs the gradient of the loss with respect to **every** weight. **Backpropagation** is the chain rule applied
layer by layer, from the output to the input, reusing what the forward pass computed. We do it by hand on the smallest
interesting network: 2 inputs, 2 sigmoid hidden units, 1 sigmoid output, squared error on one example.

In [ ]:
def sig(a):
    return 1 / (1 + np.exp(-a))


x = np.array([1.0, 0.5])
t = 1.0                                          # target
W1 = np.array([[0.1, 0.4], [-0.2, 0.3]])         # hidden layer: rows = units
b1 = np.array([0.0, 0.1])
W2 = np.array([0.7, -0.5])
b2 = 0.2

# forward pass
z1 = W1 @ x + b1
a1 = sig(z1)
z2 = W2 @ a1 + b2
a2 = sig(z2)
loss = 0.5 * (a2 - t) ** 2
print("forward :  z1 =", z1.round(4), " a1 =", a1.round(4), " z2 =", round(z2, 4), " output =", round(a2, 4), " loss =", round(loss, 5))

# backward pass: dL/dz of every layer, then the weights
d2 = (a2 - t) * a2 * (1 - a2)                    # dL/dz2
gW2 = d2 * a1
gb2 = d2
d1 = (W2 * d2) * a1 * (1 - a1)                   # dL/dz1: the error travels back through W2
gW1 = np.outer(d1, x)
gb1 = d1
print("backward:  dL/dz2 =", round(d2, 5), " dL/dz1 =", d1.round(5))

# the same gradient from JAX and from finite differences
def f(params):
    h = jax.nn.sigmoid(params["W1"] @ x + params["b1"])
    o = jax.nn.sigmoid(params["W2"] @ h + params["b2"])
    return 0.5 * (o - t) ** 2


params = {"W1": jnp.array(W1), "b1": jnp.array(b1), "W2": jnp.array(W2), "b2": jnp.array(b2)}
g = jax.grad(f)(params)
hand = {"W1": gW1, "b1": gb1, "W2": gW2, "b2": gb2}
eps = 1e-4
print(f"{'parameter':<10} {'by hand':>34} {'jax.grad':>34} {'finite differences':>34}")
for k in ("W1", "b1", "W2", "b2"):
    fd = np.zeros_like(np.asarray(params[k]), dtype=float)
    it = np.nditer(fd, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        p1 = {kk: np.array(v, dtype=float) for kk, v in params.items()}
        p2 = {kk: np.array(v, dtype=float) for kk, v in params.items()}
        p1[k][idx] += eps
        p2[k][idx] -= eps
        fd[idx] = (float(f({kk: jnp.array(v) for kk, v in p1.items()})) - float(f({kk: jnp.array(v) for kk, v in p2.items()}))) / (2 * eps)
    print(f"{k:<10} {str(np.round(hand[k], 5)):>34} {str(np.round(np.asarray(g[k]), 5)):>34} {str(fd.round(5)):>34}")

**One gradient step and a loop.** Repeating forward, backward, update drives the loss down; with a learning rate of 2 the
loss of this one example goes from 0.086 towards zero.

In [ ]:
P = {k: np.array(v, dtype=float) for k, v in params.items()}
hist = []
for _ in range(60):
    gg = jax.grad(f)({k: jnp.array(v) for k, v in P.items()})
    P = {k: P[k] - 2.0 * np.asarray(gg[k]) for k in P}
    hist.append(float(f({k: jnp.array(v) for k, v in P.items()})))
print("loss after 1, 5, 20, 60 steps:", [round(hist[i - 1], 5) for i in (1, 5, 20, 60)])

### Vanishing gradients

The backward pass multiplies by $\phi'(z)$ at every layer. A sigmoid has $\phi' \le 0.25$: after $L$ layers the gradient has
shrunk like $0.25^L$, and the first layers stop learning. ReLU has $\phi' = 1$ on the active side. Gradient norm of the first
layer for a deep MLP (width 32, 5 random draws, He initialization):

In [ ]:
def deep_grad_norms(depth, act, seed, width=32):
    key = jax.random.PRNGKey(seed)
    keys = jax.random.split(key, depth + 2)
    Ws = [jax.random.normal(keys[i], (width, width if i else 10)) * np.sqrt(2 / (width if i else 10)) for i in range(depth)]
    wo = jax.random.normal(keys[-1], (width,)) * np.sqrt(1 / width)
    xx = jax.random.normal(keys[-2], (10,))

    def net(Ws):
        h = xx
        for W in Ws:
            h = act(W @ h)
        return jnp.sum(wo * h)

    gs = jax.grad(net)(Ws)
    return [float(jnp.linalg.norm(g_)) for g_ in gs]


depths = [2, 4, 8, 16, 24, 32]
acts = {"sigmoid": jax.nn.sigmoid, "tanh": jnp.tanh, "ReLU": jax.nn.relu}
print(f"{'depth':>6}" + "".join(f"{n:>14}" for n in acts) + "     (norm of the gradient of layer 1, median of 5 draws)")
curves = {n: [] for n in acts}
for L in depths:
    row = []
    for n, a in acts.items():
        v = np.median([deep_grad_norms(L, a, s)[0] for s in range(5)])
        curves[n].append(v)
        row.append(v)
    print(f"{L:>6}" + "".join(f"{v:>14.2e}" for v in row))
for n, c in curves.items():
    plt.semilogy(depths, c, "o-", label=n)
plt.xlabel("depth (layers)")
plt.ylabel("gradient norm, first layer")
plt.legend()
plt.show()

**Counting parameters.** A dense layer with $n_\text{in}$ inputs and $n_\text{out}$ units has $n_\text{in} n_\text{out} + n_\text{out}$
parameters. The cost of one forward pass is about the number of weights (one multiply-add each); the backward pass costs about twice that.

In [ ]:
rows = []
for sizes in ([], [16], [64], [64, 64], [256, 256], [512, 512, 512]):
    net = mlp(sizes, 64, 10)
    rows.append((str(sizes) if sizes else "none", net.count_params()))
table(rows, ["hidden layers", "parameters"])
mlp([64, 64], 64, 10).summary()

## H3 — Training a network

The recipe of Class 02 applied to a non-convex loss: initialize, then repeat *minibatch → forward → loss → backward →
update*. An **epoch** is one pass over the training data. We use the handwritten digits (64 pixels, 10 classes).

The whole network in Keras is four lines: `Sequential`, `compile` (optimizer and loss), `fit`, `evaluate`.

In [ ]:
Xd_fit, Xd_val, yd_fit, yd_val = train_test_split(Xd_tr, yd_tr, test_size=0.2, stratify=yd_tr, random_state=0)
net = mlp([64, 64], 64, 10, seed=0)
t0 = time.time()
hist = fit_classifier(net, Xd_fit, yd_fit, Xd_val, yd_val, epochs=60, lr=1e-3, batch=32)
print(f"60 epochs in {time.time() - t0:.1f} s; train acc {hist.history['accuracy'][-1]:.3f}, validation acc {hist.history['val_accuracy'][-1]:.3f}, test acc {accuracy(net, Xd_te, yd_te):.3f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(hist.history["loss"], label="train")
axes[0].plot(hist.history["val_loss"], label="validation")
axes[0].set(xlabel="epoch", ylabel="cross-entropy", title="loss", yscale="log")
axes[0].legend()
axes[1].plot(hist.history["accuracy"], label="train")
axes[1].plot(hist.history["val_accuracy"], label="validation")
axes[1].set(xlabel="epoch", title="accuracy", ylim=(0.8, 1.01))
plt.tight_layout()
plt.show()

**The learning rate** is the most important hyperparameter: too small is slow, too large diverges (Class 02, again).
Final validation accuracy after 20 epochs (batch 64), for a log grid of learning rates and three optimizers:

In [ ]:
lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
rows = []
for lr_ in lrs:
    row = [lr_]
    for name in ("adam", "sgd", "sgd_m"):
        net = mlp([64, 64], 64, 10, seed=0)
        opt = {"adam": keras.optimizers.Adam(lr_), "sgd": keras.optimizers.SGD(lr_), "sgd_m": keras.optimizers.SGD(lr_, momentum=0.9)}[name]
        net.compile(opt, "sparse_categorical_crossentropy", metrics=["accuracy"])
        h = net.fit(Xd_fit, yd_fit, epochs=20, batch_size=64, validation_data=(Xd_val, yd_val), verbose=0)
        row.append(h.history["val_accuracy"][-1])
    rows.append(tuple(row))
table(rows, ["learning rate", "Adam", "SGD", "SGD + momentum 0.9"])

**Batch size.** A small batch gives a noisy but cheap gradient (many updates per epoch); a large batch a precise one (few
updates). Validation accuracy after 5 epochs and the wall-clock time of 30 epochs:

In [ ]:
rows = []
for bs in (8, 32, 128, 512, len(yd_fit)):
    net = mlp([64, 64], 64, 10, seed=0)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    h5 = net.fit(Xd_fit, yd_fit, epochs=5, batch_size=bs, validation_data=(Xd_val, yd_val), verbose=0)
    net2 = mlp([64, 64], 64, 10, seed=0)
    net2.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy")
    t0 = time.time()
    net2.fit(Xd_fit, yd_fit, epochs=20, batch_size=bs, verbose=0)
    rows.append((bs, int(np.ceil(len(yd_fit) / bs)), h5.history["val_accuracy"][-1], time.time() - t0))
table(rows, ["batch size", "updates per epoch", "val acc after 5 epochs", "time of 20 epochs (s)"])

**Initialization.** If all weights start equal, all hidden units compute the same thing and receive the same gradient: they
stay identical forever (**symmetry**). Random weights break it; their *scale* matters: Glorot and He scale by the number
of inputs so that signals neither explode nor vanish.

In [ ]:
inits = {"zeros": "zeros", "normal sd 0.01": keras.initializers.RandomNormal(stddev=0.01), "normal sd 1": keras.initializers.RandomNormal(stddev=1.0),
         "Glorot uniform": "glorot_uniform", "He normal": "he_normal"}
rows = []
for name, init in inits.items():
    net = mlp([32, 32], 64, 10, init=init, seed=0)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    h = net.fit(Xd_fit, yd_fit, epochs=10, batch_size=32, validation_data=(Xd_val, yd_val), verbose=0)
    W = net.layers[0].get_weights()[0]
    rows.append((name, h.history["loss"][0], h.history["val_accuracy"][-1], len({tuple(np.round(col, 5)) for col in W.T})))
table(rows, ["initializer", "loss, epoch 1", "val acc, epoch 10", "distinct hidden units (of 32)"])

**Seeds.** A network is trained from a random start on a non-convex loss: the result is a random variable (Class 01).
Ten seeds, same data, same hyperparameters:

In [ ]:
accs = []
for seed in range(10):
    net = mlp([64, 64], 64, 10, seed=seed)
    fit_classifier(net, Xd_fit, yd_fit, epochs=30, lr=1e-3, batch=64)
    accs.append(accuracy(net, Xd_te, yd_te))
print("test accuracy over 10 seeds:", np.round(accs, 3))
print(f"mean {np.mean(accs):.4f}, sd {np.std(accs, ddof=1):.4f}, min {np.min(accs):.3f}, max {np.max(accs):.3f}  → report the spread, not the best seed")

### The same network in plain JAX (the lab exercise)

Keras hides three things that the lab asks you to write: the **parameters** (a list of `(W, b)` pairs, a *pytree*), the
**forward pass** (a loop over layers) and the **loss** (cross-entropy from logits with `logsumexp`). The gradient is `jax.grad`.

In [ ]:
def init_params(key, sizes):
    params = []
    for k, (m, n) in zip(jax.random.split(key, len(sizes) - 1), zip(sizes[:-1], sizes[1:])):
        params.append({"W": jax.random.normal(k, (m, n)) * jnp.sqrt(2.0 / m), "b": jnp.zeros(n)})     # He initialization
    return params


def forward(params, X):
    h = X
    for layer in params[:-1]:
        h = jax.nn.relu(h @ layer["W"] + layer["b"])
    return h @ params[-1]["W"] + params[-1]["b"]                                                      # logits


def xent(params, X, y):
    logits = forward(params, X)
    return jnp.mean(jax.nn.logsumexp(logits, axis=1) - jnp.take_along_axis(logits, y[:, None], axis=1)[:, 0])


@jax.jit
def adam_step(params, state, X, y, t, lr=1e-3):
    loss, g = jax.value_and_grad(xent)(params, X, y)
    m, v = state
    m = jax.tree.map(lambda a, b: 0.9 * a + 0.1 * b, m, g)
    v = jax.tree.map(lambda a, b: 0.999 * a + 0.001 * b**2, v, g)
    params = jax.tree.map(lambda p, a, b: p - lr * (a / (1 - 0.9**t)) / (jnp.sqrt(b / (1 - 0.999**t)) + 1e-8), params, m, v)
    return params, (m, v), loss


Xj, yj = jnp.asarray(Xd_fit, dtype=jnp.float32), jnp.asarray(yd_fit)
params = init_params(jax.random.PRNGKey(0), [64, 64, 64, 10])
state = (jax.tree.map(jnp.zeros_like, params), jax.tree.map(jnp.zeros_like, params))
rng = np.random.default_rng(0)
t = 0
t0 = time.time()
for epoch in range(30):
    perm = rng.permutation(len(yj))
    for i in range(0, len(perm), 32):
        t += 1
        idx = perm[i : i + 32]
        params, state, loss = adam_step(params, state, Xj[idx], yj[idx], t)
pred = np.asarray(forward(params, jnp.asarray(Xd_te, dtype=jnp.float32)).argmax(1))
print(f"plain JAX: 30 epochs in {time.time() - t0:.1f} s, test accuracy {np.mean(pred == yd_te):.3f} (Keras, same architecture: mean {np.mean(accs):.3f})")
print("parameters:", sum(int(np.prod(p['W'].shape)) + int(np.prod(p['b'].shape)) for p in params))

## H4 — Capacity and regularization

A network with more parameters than examples can **memorize** the training set (Class 01: overfitting). The tools to control
it are the ones of Classes 03-04, plus two that are specific to networks:

* **weight decay** ($L_2$ penalty on the weights, a Gaussian prior: ridge, Class 04);
* **early stopping**: stop when the validation loss stops improving (the number of epochs is a hyperparameter);
* **dropout**: during training each hidden unit is switched off with probability $p$, so no unit can rely on a partner.

A small noisy problem: 100 moons with noise 0.4 (a lot of overlap), 2000 test points.

In [ ]:
Xs, ys = make_moons(100, noise=0.4, random_state=3)
Xs_te, ys_te = make_moons(4000, noise=0.4, random_state=4)
Xs_tr, Xs_va, ys_tr, ys_va = train_test_split(Xs, ys, test_size=0.3, stratify=ys, random_state=0)
big = mlp([128, 128], 2, 1, out="sigmoid", seed=0)
h = fit_classifier(big, Xs_tr, ys_tr, Xs_va, ys_va, epochs=300, lr=0.005, batch=16)
print(f"no regularization: train acc {h.history['accuracy'][-1]:.3f}, validation acc {h.history['val_accuracy'][-1]:.3f}, test acc {accuracy(big, Xs_te, ys_te):.3f}; parameters {big.count_params()} for {len(ys_tr)} examples")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(h.history["loss"], label="train loss")
ax.plot(h.history["val_loss"], label="validation loss")
ax.set(xlabel="epoch", ylabel="cross-entropy")
ax.axvline(int(np.argmin(h.history["val_loss"])), color="k", ls=":", label="best epoch")
ax.legend()
plt.show()
print(f"best validation loss at epoch {int(np.argmin(h.history['val_loss'])) + 1}; at epoch 300 it is {h.history['val_loss'][-1] / min(h.history['val_loss']):.1f} times larger")

**Four ways to regularize, side by side** (the same architecture, 3 seeds, median test accuracy):

In [ ]:
def run(seed, l2=0.0, dropout=0.0, early=False, epochs=150):
    net = mlp([128, 128], 2, 1, out="sigmoid", l2=l2, dropout=dropout, seed=seed)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)] if early else None
    fit_classifier(net, Xs_tr, ys_tr, Xs_va, ys_va, epochs=epochs, lr=0.005, batch=32, callbacks=cb)
    return net


configs = {
    "none": {},
    "weight decay 1e-3": {"l2": 1e-3},
    "weight decay 1e-2": {"l2": 1e-2},
    "dropout 0.3": {"dropout": 0.3},
    "dropout 0.5": {"dropout": 0.5},
    "early stopping (patience 20)": {"early": True},
    "dropout 0.3 + early stopping": {"dropout": 0.3, "early": True},
}
rows = []
nets = {}
for name, kw in configs.items():
    res = []
    for seed in range(3):
        net = run(seed, **kw)
        res.append((accuracy(net, Xs_tr, ys_tr), accuracy(net, Xs_te, ys_te)))
        nets[name] = net
    r = np.array(res)
    rows.append((name, float(np.median(r[:, 0])), float(np.median(r[:, 1])), float(r[:, 1].min()), float(r[:, 1].max())))
lr_ = LogisticRegression().fit(Xs_tr, ys_tr)
rows.append(("logistic regression (linear)", lr_.score(Xs_tr, ys_tr), lr_.score(Xs_te, ys_te), np.nan, np.nan))
table(rows, ["regularization", "train acc", "test acc (median of 3)", "test acc min", "test acc max"])
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, name in zip(axes, ("none", "weight decay 1e-2", "dropout 0.5", "early stopping (patience 20)")):
    boundary(ax, lambda P, n=nets[name]: n.predict(P, verbose=0)[:, 0], Xs_tr, ys_tr, name)
plt.tight_layout()
plt.show()

**Width and depth.** More units or more layers means more capacity. With early stopping and a validation set, bigger is
rarely worse; without regularization the gap between train and test grows.

In [ ]:
rows = []
for sizes in ([4], [16], [64], [256], [16, 16], [64, 64], [64, 64, 64], [256, 256]):
    r = []
    for seed in range(2):
        net = mlp(sizes, 2, 1, out="sigmoid", seed=seed)
        fit_classifier(net, Xs_tr, ys_tr, epochs=120, lr=0.005, batch=32)
        r.append((accuracy(net, Xs_tr, ys_tr), accuracy(net, Xs_te, ys_te)))
    r = np.array(r)
    rows.append((str(sizes), net.count_params(), float(np.median(r[:, 0])), float(np.median(r[:, 1]))))
table(rows, ["architecture", "parameters", "train acc", "test acc"])

## H5 — A network for regression

For a number, the output layer has **one linear unit** and the loss is the **mean squared error** (a Gaussian noise
assumption, Class 04). The target is standardized, as in V5. Two datasets: concrete strength (1030 rows) and California
housing (20640 rows).

In [ ]:
def reg_mlp(sizes, n_in, seed=0, l2=0.0):
    return mlp(sizes, n_in, 1, out=None, l2=l2, seed=seed)


def cv_rmse_reg(make, X, y, folds=5, epochs=100, batch=64, lr=3e-3, seed=0):
    out = []
    bins = np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, bins):
        sc = StandardScaler().fit(X[a])
        mu, sd = y[a].mean(), y[a].std()
        model = make()
        if isinstance(model, keras.Model):
            model.compile(keras.optimizers.Adam(lr), "mse")
            model.fit(sc.transform(X[a]), (y[a] - mu) / sd, epochs=epochs, batch_size=batch, verbose=0)
            pred = model.predict(sc.transform(X[b]), verbose=0)[:, 0]
        else:
            pred = model.fit(sc.transform(X[a]), (y[a] - mu) / sd).predict(sc.transform(X[b]))
        out.append(np.sqrt(np.mean((pred * sd + mu - y[b]) ** 2)))
    return np.mean(out), np.std(out)


Xc, yc = load_xy("concrete", "strength")
models = {
    "ridge": lambda: Ridge(alpha=1.0),
    "SVR rbf (C=30, gamma=0.2)": lambda: SVR(C=30, gamma=0.2, epsilon=0.05),
    "MLP [16]": lambda: reg_mlp([16], 8),
    "MLP [64, 64]": lambda: reg_mlp([64, 64], 8),
    "MLP [64, 64], weight decay 1e-3": lambda: reg_mlp([64, 64], 8, l2=1e-3),
    "MLP [256, 256, 256]": lambda: reg_mlp([256, 256, 256], 8),
}
rows = [(name, *cv_rmse_reg(make, Xc, yc)) for name, make in models.items()]
table(rows, ["model (concrete, 5-fold)", "RMSE (MPa)", "sd over folds"])

**Other datasets, same protocol.** Abalone (rings from shell measurements, one categorical feature), auto-mpg (6 missing
values) and the diabetes data: the baseline that predicts the mean, ridge, SVR and two MLPs. On small noisy data the
network is not clearly better than the linear model: the data limit every learner.

In [ ]:
rows = []
for dname, target in (("abalone", "rings"), ("auto_mpg", "mpg"), ("diabetes", "target")):
    Xr, yr = load_table(dname, target)
    yr = yr.astype(float)
    base = cv_rmse_reg(lambda: Ridge(alpha=1e9), Xr, yr)[0]
    row = [dname, base]
    for make in (lambda: Ridge(alpha=1.0), lambda: SVR(C=3, epsilon=0.1), lambda: reg_mlp([32], Xr.shape[1], l2=1e-3), lambda: reg_mlp([64, 64], Xr.shape[1], l2=1e-3)):
        row.append(cv_rmse_reg(make, Xr, yr, epochs=60)[0])
    rows.append(tuple(row))
table(rows, ["dataset (5-fold RMSE)", "mean", "ridge", "SVR rbf", "MLP [32]", "MLP [64, 64]"])

**Learning curve and the target scale.** Training with the raw target (values 2-82 MPa) instead of the standardized one:
the first updates are huge and the loss starts at about $30^2$; the network recovers slowly or not at all.

In [ ]:
sc = StandardScaler().fit(Xc)
Xs_c = sc.transform(Xc)
mu, sd = yc.mean(), yc.std()
tr, te = train_test_split(np.arange(len(yc)), test_size=0.25, random_state=0)
curves = {}
for label, (t_tr, t_te) in {"standardized target": ((yc[tr] - mu) / sd, (yc[te] - mu) / sd), "raw target (MPa)": (yc[tr], yc[te])}.items():
    net = reg_mlp([64, 64], 8)
    net.compile(keras.optimizers.Adam(2e-3), "mse")
    h = net.fit(Xs_c[tr], t_tr, epochs=200, batch_size=64, validation_data=(Xs_c[te], t_te), verbose=0)
    scale = sd if "standardized" in label else 1.0
    curves[label] = np.sqrt(np.array(h.history["val_loss"])) * scale
    print(f"{label:<22}: validation RMSE after 5, 50, 200 epochs: {curves[label][4]:.2f}, {curves[label][49]:.2f}, {curves[label][-1]:.2f} MPa")
for k, v in curves.items():
    plt.plot(v, label=k)
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("validation RMSE (MPa)")
plt.legend()
plt.show()

**A larger problem.** California housing (20640 districts, log of the skewed features as in Class 03): at this size the
network overtakes the linear model clearly, and SVR is already too slow to be practical without subsampling.

In [ ]:
Xh, yh = load_xy("california_housing", "MedHouseVal")
names = pl.read_csv(DATA / "california_housing.csv.zst").drop("MedHouseVal").columns
for j, nme in enumerate(names):
    if nme in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
        Xh[:, j] = np.log(Xh[:, j])
tr, te = train_test_split(np.arange(len(yh)), test_size=0.2, random_state=0)
sc = StandardScaler().fit(Xh[tr])
rows = []
r = Ridge(1.0).fit(sc.transform(Xh[tr]), yh[tr])
rows.append(("ridge", np.sqrt(np.mean((r.predict(sc.transform(Xh[te])) - yh[te]) ** 2)), 0.0))
for sizes in ([64], [64, 64], [128, 128, 128]):
    net = reg_mlp(sizes, 8)
    net.compile(keras.optimizers.Adam(1e-3), "mse")
    t0 = time.time()
    net.fit(sc.transform(Xh[tr]), yh[tr], epochs=40, batch_size=256, verbose=0, validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])
    rows.append((f"MLP {sizes}", np.sqrt(np.mean((net.predict(sc.transform(Xh[te]), verbose=0)[:, 0] - yh[te]) ** 2)), time.time() - t0))
sub = np.random.default_rng(0).choice(tr, 3000, replace=False)
t0 = time.time()
svr = SVR(C=3, gamma=0.2).fit(sc.transform(Xh[sub]), yh[sub])
rows.append(("SVR rbf (3000 rows only)", np.sqrt(np.mean((svr.predict(sc.transform(Xh[te])) - yh[te]) ** 2)), time.time() - t0))
table(rows, ["model", "test RMSE (100k USD)", "train time (s)"])
print(f"baseline (predict the training mean): RMSE {np.sqrt(np.mean((yh[tr].mean() - yh[te]) ** 2)):.3f}")

## H6 — Which learner? One protocol, three datasets

Logistic regression (Class 03), the RBF SVM (V4) and an MLP, on the same splits and the same scaling, medians over 5
repeated stratified 75/25 splits. The comparison of two learners uses the **corrected resampled t-test** of Class 01
(Nadeau and Bengio): the repeated splits are not independent, so the variance is inflated by $1/J + n_\text{test}/n_\text{train}$.

In [ ]:
def load_sets():
    out = {}
    for name, target, f in (
        ("breast_cancer", "target", lambda X, y: (X, y)),
        ("spambase", "spam", lambda X, y: (np.log1p(X), y)),
        ("digits", "target", lambda X, y: (X / 16.0, y)),
        ("wine (quality >= 6)", "quality", lambda X, y: (X, (y >= 6).astype(int))),
        ("heart_disease", "num", lambda X, y: (X, (y == ">50_1").astype(int))),
    ):
        key = name.split(" (")[0] if name.startswith("wine") else name
        out[name] = f(*load_table(key if key != "wine" else "wine_quality", target))
    return out


def make_mlp(n_in, n_out):
    def build():
        net = mlp([64, 64], n_in, n_out if n_out > 2 else 1, out="softmax" if n_out > 2 else "sigmoid", dropout=0.2, seed=0)
        return net
    return build


results = {}
timing = {}
for dname, (X, y) in load_sets().items():
    n_out = len(np.unique(y))
    for rep in range(5):
        Xa, Xb_, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=rep)
        sc = StandardScaler().fit(Xa)
        Xa, Xb_ = sc.transform(Xa), sc.transform(Xb_)
        t = time.time()
        a_lr = LogisticRegression(C=1, max_iter=5000).fit(Xa, ya).score(Xb_, yb_)
        t_lr = time.time() - t
        t = time.time()
        a_svm = SVC(C=3, gamma="scale").fit(Xa, ya).score(Xb_, yb_)
        t_svm = time.time() - t
        t = time.time()
        net = make_mlp(Xa.shape[1], n_out)()
        fit_classifier(net, Xa, ya, epochs=30, lr=1e-3, batch=64)
        a_mlp = accuracy(net, Xb_, yb_)
        t_mlp = time.time() - t
        for k, a, tt in (("logistic", a_lr, t_lr), ("SVM rbf", a_svm, t_svm), ("MLP", a_mlp, t_mlp)):
            results.setdefault((dname, k), []).append(a)
            timing.setdefault((dname, k), []).append(tt)
rows = [(d, k, float(np.median(v)), float(np.std(v, ddof=1)), float(np.median(timing[(d, k)]))) for (d, k), v in results.items()]
table(rows, ["dataset", "model", "median accuracy", "sd", "fit time (s)"])

**Is the difference real?** The corrected resampled t-test on the paired accuracies (SVM minus MLP, then SVM minus logistic):

In [ ]:
results_names = list(dict.fromkeys(d for d, _ in results))


def corrected_ttest(a, b, n_test_over_n_train=1 / 3):
    d = np.asarray(a) - np.asarray(b)
    J = len(d)
    var = np.var(d, ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / J + n_test_over_n_train) * var)
    return float(t), float(2 * stats.t.sf(abs(t), J - 1))


rows = []
for dname in results_names:
    for a, b in (("SVM rbf", "MLP"), ("SVM rbf", "logistic"), ("MLP", "logistic")):
        t, p = corrected_ttest(results[(dname, a)], results[(dname, b)])
        rows.append((dname, f"{a} - {b}", float(np.mean(results[(dname, a)]) - np.mean(results[(dname, b)])), t, p))
table(rows, ["dataset", "comparison", "mean difference", "t", "p"])
fig, axes = plt.subplots(1, len(results_names), figsize=(16, 3.4))
for ax, dname in zip(axes, results_names):
    ax.boxplot([results[(dname, k)] for k in ("logistic", "SVM rbf", "MLP")], tick_labels=["logistic", "SVM rbf", "MLP"])
    ax.set_title(dname)
axes[0].set_ylabel("test accuracy (5 splits)")
plt.tight_layout()
plt.show()

## Summary

* A neuron is a linear model followed by a non-linear activation; layers compose them; the hidden layers **learn the
  features** that the kernel of the SVM fixed in advance.
* **Backpropagation** is the chain rule, layer by layer; `jax.grad` (and Keras) do it for any network.
* Training: learning rate first, an adaptive optimizer, minibatches, an initialization that preserves the signal scale,
  several seeds.
* Regularize: weight decay, dropout, early stopping; judge on validation data.
* No learner wins everywhere: logistic regression, SVM and MLP are close on small tabular data; networks pay off when the
  data are many and the function is complex.